# Automatic differentiation

In [1]:
import numpy as np
import scibmad as sb
# Julia's automatic differentiation packages, used through DifferentiationInterface:
DI = sb.julia.package("DifferentiationInterface", install=True)
for name in ("ForwardDiff", "ReverseDiff", "FiniteDiff", "FiniteDifferences"):
    sb.julia.package(name, install=True)
FiniteDifferences = sb.julia.package("FiniteDifferences")

In [2]:
K1 = 0.36
K2 = 0.1

qf = sb.Quadrupole(L=0.5, Kn1=K1)
sf = sb.Sextupole(L=0.2, Kn2=K2)
d = sb.Drift(L=0.6)
b = sb.SBend(L=6.0, angle=np.pi/132)
qd = sb.Quadrupole(L=0.5, Kn1=-K1)
sd = sb.Sextupole(L=0.2, Kn2=-K2)
fodo = sb.Beamline([qf, d, sf, b, sd, d, qd], species_ref=sb.Species("electron"), E_ref=18e9)

In [3]:
# Returns exit coordinates for 1 particle (the coordinates are Julia arrays of the numbers
# each AD package differentiates with: the function is called from Julia)
def track_a_particle(v0, beamline):
    res = sb.track(beamline, v0=sb.reshape(v0, 1, 6))
    return res.v(turn=1, particle=1)

x0 = np.array([1e-2, 0.0, 1e-2, 0.0, 0.0, 0.0])

In [4]:
# Forward AD using ForwardDiff.jl:
backend = sb.AutoForwardDiff()
prep_FD = DI.prepare_jacobian(track_a_particle, backend, x0, DI.Constant(fodo))
J_FD = DI.jacobian(track_a_particle, prep_FD, backend, x0, DI.Constant(fodo))
J_FD

array([[-5.01415875e-01,  8.58615359e+00,  4.78046063e-04,
        -3.15259977e-03,  0.00000000e+00,  1.22033492e-01],
       [-2.57407434e-01,  2.41343769e+00, -6.56685953e-04,
        -3.60534875e-03,  0.00000000e+00,  4.39660781e-02],
       [ 1.47842013e-03, -1.93047102e-03,  2.41669541e+00,
         8.59143168e+00,  0.00000000e+00, -1.37771462e-02],
       [ 2.98994081e-05, -2.60515304e-03, -2.57076863e-01,
        -5.00126953e-01,  0.00000000e+00,  2.54320575e-03],
       [-9.37121029e-03, -8.29384701e-02, -2.70552363e-03,
        -1.52608222e-02,  1.00000000e+00, -4.82958969e-04],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  1.00000000e+00]])

In [5]:
# Taylor AD using GTPSA.jl:
backend = sb.AutoGTPSA()
prep_GTPSA = DI.prepare_jacobian(track_a_particle, backend, x0, DI.Constant(fodo))
J_GTPSA = DI.jacobian(track_a_particle, prep_GTPSA, backend, x0, DI.Constant(fodo))
J_GTPSA

array([[-5.01415875e-01,  8.58615359e+00,  4.78046063e-04,
        -3.15259977e-03,  0.00000000e+00,  1.22033492e-01],
       [-2.57407434e-01,  2.41343769e+00, -6.56685953e-04,
        -3.60534875e-03,  0.00000000e+00,  4.39660781e-02],
       [ 1.47842013e-03, -1.93047102e-03,  2.41669541e+00,
         8.59143168e+00,  0.00000000e+00, -1.37771462e-02],
       [ 2.98994081e-05, -2.60515304e-03, -2.57076863e-01,
        -5.00126953e-01,  0.00000000e+00,  2.54320575e-03],
       [-9.37121029e-03, -8.29384701e-02, -2.70552363e-03,
        -1.52608222e-02,  1.00000000e+00, -4.82958969e-04],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  1.00000000e+00]])

In [6]:
# Reverse AD using ReverseDiff.jl:
backend = sb.AutoReverseDiff()
prep_RD = DI.prepare_jacobian(track_a_particle, backend, x0, DI.Constant(fodo))
J_RD = DI.jacobian(track_a_particle, prep_RD, backend, x0, DI.Constant(fodo))
J_RD

array([[-5.01415875e-01,  8.58615359e+00,  4.78046063e-04,
        -3.15259977e-03,  0.00000000e+00,  1.22033492e-01],
       [-2.57407434e-01,  2.41343769e+00, -6.56685953e-04,
        -3.60534875e-03,  0.00000000e+00,  4.39660781e-02],
       [ 1.47842013e-03, -1.93047102e-03,  2.41669541e+00,
         8.59143168e+00,  0.00000000e+00, -1.37771462e-02],
       [ 2.98994081e-05, -2.60515304e-03, -2.57076863e-01,
        -5.00126953e-01,  0.00000000e+00,  2.54320575e-03],
       [-9.37121029e-03, -8.29384701e-02, -2.70552363e-03,
        -1.52608222e-02,  1.00000000e+00, -4.82958969e-04],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  1.00000000e+00]])

In [7]:
# Finite differencing using FiniteDiff.jl:
backend = sb.AutoFiniteDiff()
prep_fdif1 = DI.prepare_jacobian(track_a_particle, backend, x0, DI.Constant(fodo))
J_fdif1 = DI.jacobian(track_a_particle, prep_fdif1, backend, x0, DI.Constant(fodo))
J_fdif1

array([[-5.01415874e-01,  8.58615359e+00,  4.78045549e-04,
        -3.15261295e-03,  0.00000000e+00,  1.22033491e-01],
       [-2.57407434e-01,  2.41343770e+00, -6.56686607e-04,
        -3.60535920e-03,  0.00000000e+00,  4.39660785e-02],
       [ 1.47841917e-03, -1.93047128e-03,  2.41669541e+00,
         8.59143168e+00,  0.00000000e+00, -1.37771466e-02],
       [ 2.98994710e-05, -2.60515479e-03, -2.57076863e-01,
        -5.00126955e-01,  0.00000000e+00,  2.54320569e-03],
       [-9.37117843e-03, -8.29384893e-02, -2.70551899e-03,
        -1.52608881e-02,  1.00000000e+00, -4.82933804e-04],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  1.00000000e+00]])

In [8]:
# Finite differencing using FiniteDifferences.jl:
backend = sb.AutoFiniteDifferences(fdm=FiniteDifferences.central_fdm(3, 1))
prep_fdif2 = DI.prepare_jacobian(track_a_particle, backend, x0, DI.Constant(fodo))
J_fdif2 = DI.jacobian(track_a_particle, prep_fdif2, backend, x0, DI.Constant(fodo))
J_fdif2

array([[-5.01415875e-01,  8.58615359e+00,  4.78046070e-04,
        -3.15259978e-03,  0.00000000e+00,  1.22033492e-01],
       [-2.57407434e-01,  2.41343769e+00, -6.56685950e-04,
        -3.60534875e-03,  0.00000000e+00,  4.39660781e-02],
       [ 1.47842014e-03, -1.93047102e-03,  2.41669541e+00,
         8.59143168e+00,  0.00000000e+00, -1.37771462e-02],
       [ 2.98994081e-05, -2.60515304e-03, -2.57076863e-01,
        -5.00126953e-01,  0.00000000e+00,  2.54320575e-03],
       [-9.37120997e-03, -8.29384696e-02, -2.70552360e-03,
        -1.52608222e-02,  1.00000000e+00, -4.82959532e-04],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  1.00000000e+00]])

In [9]:
# Show they're all equal (with Julia's isapprox, as ≈ in Julia: relative to the norms)
all(sb.isapprox(J, J_FD) for J in (J_GTPSA, J_RD, J_fdif1, J_fdif2))

True